[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# LISTEN and NOTIFY


## What you will be able to do

Have the database tell your program that something happened, instead of asking it every second.
Write the listener with psycopg and with asyncpg, and say why the psycopg one hears nothing at all
without `autocommit`. Put a trigger on a table so an `INSERT` is what sends the message. Say what a
channel name does to its capital letters, and what the eight thousand byte payload limit means for
what you should put in one. And recognize the three ways a listener can be running, connected, and
still never hear anything.


## The idea

### The problem

A program that wants to know when a row arrives has two options. It can ask, over and over, which is
a query per second per process forever and still up to a second late. Or the database can tell it,
which is what `LISTEN` and `NOTIFY` are, and which costs nothing while nothing is happening.

The catch is that a listener is a connection sitting still, and almost everything this guide taught
you about connections assumed they were being used. The rules change, and the failures are silent.

### What it is

`LISTEN channel` says this session wants messages on a name. `NOTIFY channel, 'payload'` sends one,
from any session, to every session listening. The name is an identifier and the payload is a short
string. That is the whole feature.

### Why autocommit is not optional

A notification is delivered at commit. A listener that is inside an open transaction is looking at a
snapshot from before the notification existed, so psycopg, which opens a transaction for you, hears
nothing until that transaction ends. asyncpg has no implicit transaction, so the problem never
arises there, which is one of the few places its model is simply easier.

### Where this shows up

A job queue that wants to start work the moment a row is inserted. A cache that wants to know when
to drop an entry. Anything where a second of delay is worse than the complexity of a listener.

### What this notebook covers

`LISTEN`, `NOTIFY` and `pg_notify`, from both drivers. `notifies` with a timeout, and the handler
callback. asyncpg's `add_listener`. A trigger, so that writing a row is the notification. The
channel name's capital letters and the payload's size limit. Then the three silences: no autocommit,
the wrong case, and a listener whose cell ended.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import threading

import psycopg


def notify_shortly():
    # another connection, sending one notification a moment from now
    with psycopg.connect("dbname=guide", autocommit=True) as sender:
        sender.execute("NOTIFY jobs, 'a row landed'")


for autocommit in (False, True):
    listener = psycopg.connect("dbname=guide", autocommit=autocommit)
    listener.execute("LISTEN jobs")

    threading.Timer(0.3, notify_shortly).start()
    heard = list(listener.notifies(timeout=2, stop_after=1))       # never wait without a timeout

    print(f"autocommit={autocommit!s:5} heard {len(heard)}:", [n.payload for n in heard])
    listener.close()
```

```
autocommit=False heard 0: []
autocommit=True  heard 1: ['a row landed']
```

The two halves differ by one argument. Both listeners ran `LISTEN jobs`, both were connected, both
waited two seconds, and the notification was sent to both. The first one heard nothing, raised
nothing, and logged nothing, which is what makes this worth a notebook.


## Setup

Twelve imports, both drivers, the server, and four helpers.

- `psycopg` with `errors` and `sql`, which is needed once for a payload too big to parameterize
- `asyncpg` is the other listener, and `asyncio` runs it
- `threading` sends a notification a moment from now, `time` measures, `warnings` catches one
- `subprocess`, `sys`, `os`, `getpass` stand the server up with `version` and `PackageNotFoundError`

`send` notifies from another connection, optionally after a delay. `heard` waits with a timeout,
which every wait in this notebook has, because a cell waiting for a message that is never coming is
a notebook that has to be interrupted. `subscribe` opens a listener of its own, so that no section
picks up messages meant for another one, and `listening` asks the server what a connection actually
subscribed to.


In [1]:
import asyncio
import getpass
import os
import subprocess
import sys
import threading
import time
import warnings
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import errors, sql

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

def send(channel, payload="", after=0.0):
    """Notify from another connection, optionally a moment from now."""
    def once():
        with psycopg.connect("dbname=guide", autocommit=True) as conn:
            conn.execute("SELECT pg_notify(%s, %s)", (channel, payload))

    if after:
        threading.Timer(after, once).start()
    else:
        once()


def heard(listener, seconds=3, count=1):
    """Wait for notifications, with a timeout, because a cell that waits forever is a hung notebook."""
    return [(note.channel, note.payload) for note in
            listener.notifies(timeout=seconds, stop_after=count)]


def subscribe(*channels):
    """A connection of its own, already listening, so no section hears another section's backlog."""
    conn = psycopg.connect("dbname=guide", autocommit=True)         # autocommit, always
    for channel in channels:
        conn.execute(sql.SQL("LISTEN {}").format(sql.Identifier(channel)))
    return conn


def listening(conn):
    """What the server thinks this connection asked for, which is not always what you typed."""
    return [row[0] for row in conn.execute("SELECT * FROM pg_listening_channels()").fetchall()]


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


## Worked examples

### A listener, and something to hear

The listener needs `autocommit=True`, and then it is three lines:


In [2]:
listener = psycopg.connect("dbname=guide", autocommit=True)
listener.execute("LISTEN jobs")

send("jobs", "the first one", after=0.3)
start = time.perf_counter()
print("heard:", heard(listener), f"after {time.perf_counter() - start:.1f}s")


heard: [('jobs', 'the first one')] after 0.3s


Three tenths of a second, which is when it was sent. The listener was not polling: `notifies` waits
on the socket and returns the moment something arrives.

`pg_notify` and `NOTIFY` are the same thing, and the difference matters:


In [3]:
with psycopg.connect("dbname=guide", autocommit=True) as sender:
    sender.execute("NOTIFY jobs, 'from the statement'")              # channel is an identifier
    sender.execute("SELECT pg_notify(%s, %s)", ("jobs", "from the function"))

print("both arrived:", heard(listener, count=2))


both arrived: [('jobs', 'from the statement'), ('jobs', 'from the function')]


`NOTIFY` is a statement, so its channel is an identifier and cannot be a parameter. `pg_notify` is
an ordinary function, so both arguments can be. Use the function whenever the channel or the payload
comes from a variable, which is almost always.

### Several channels, and which one spoke

One connection can listen to as many as it likes, and every notification says which it came from:


In [4]:
listener.execute("LISTEN cache_drops")
print("subscribed to:", listening(listener))

send("cache_drops", "user:41", after=0.1)
send("jobs", "resize the image", after=0.2)
print("heard:", heard(listener, count=2))


subscribed to: ['jobs', 'cache_drops']
heard: [('cache_drops', 'user:41'), ('jobs', 'resize the image')]


That is why a `Notify` carries its channel: one listener and one loop can serve every kind of event
in the program, and the channel is how the loop decides what to do.

`UNLISTEN` takes one back, and `UNLISTEN *` takes them all:


In [5]:
listener.execute("UNLISTEN cache_drops")
print("subscribed to:", listening(listener))

send("cache_drops", "nobody is listening")
print("heard:", heard(listener, seconds=1), "<- an empty list is the timeout expiring")


subscribed to: ['jobs']
heard: [] <- an empty list is the timeout expiring


A notification with no listener is discarded by the server. There is no queue, nothing is stored,
and this is the single most important property of the feature: **a listener that was not connected
missed it, permanently**.

### A trigger, so the row itself is the message

This is what the feature is actually for. Writing a row becomes the notification:


In [6]:
with psycopg.connect("dbname=guide", autocommit=True) as setup:
    setup.execute("DROP TABLE IF EXISTS queue")
    setup.execute("CREATE TABLE queue (id bigserial PRIMARY KEY, task text)")
    setup.execute("""
        CREATE OR REPLACE FUNCTION announce_queue() RETURNS trigger AS $$
        BEGIN
            PERFORM pg_notify('queue', NEW.id || ':' || NEW.task);
            RETURN NEW;
        END;
        $$ LANGUAGE plpgsql""")
    setup.execute("""
        CREATE OR REPLACE TRIGGER queue_announced AFTER INSERT ON queue
        FOR EACH ROW EXECUTE FUNCTION announce_queue()""")

listener.execute("LISTEN queue")

with psycopg.connect("dbname=guide") as writer:                     # an ordinary transaction
    writer.execute("INSERT INTO queue (task) VALUES ('resize')")
    print("before the commit:", heard(listener, seconds=1))
    writer.commit()

print("after the commit: ", heard(listener, seconds=3))


before the commit: []
after the commit:  [('queue', '1:resize')]


Two things happened there. The insert alone delivered nothing, because a notification is queued
inside the transaction and sent when it commits, which is the behavior that makes this safe: you
will never be told about a row that was rolled back.

And the program that writes rows did not have to know a listener exists. That is the argument for
putting the `NOTIFY` in a trigger rather than in application code.

### asyncpg, which has no autocommit to forget

A callback rather than a loop, and no transaction problem at all:


In [7]:
conn = await asyncpg.connect(database="guide")
arrived = []


def collect(connection, pid, channel, payload):
    """Four arguments, always these four, and it must not block."""
    arrived.append((channel, payload))


await conn.add_listener("queue", collect)
send("queue", "written by hand")
await asyncio.sleep(0.5)                                            # let the loop deliver it
print("the callback saw:", arrived)

await conn.remove_listener("queue", collect)
send("queue", "and this one goes nowhere")
await asyncio.sleep(0.5)
print("after remove_listener:", arrived)


the callback saw: [('queue', 'written by hand')]
after remove_listener: [('queue', 'written by hand')]


`add_listener` runs the `LISTEN` and registers a callback, which the event loop calls when something
arrives. The four arguments are always the same: the connection, the sending backend's process id,
the channel, and the payload.

The `await asyncio.sleep` is the part to understand rather than copy. Nothing is delivered while
your code is not giving the event loop a turn, so a real listener spends its life inside something
that waits:


In [8]:
async def listen_until(channel, count, seconds):
    """A listener with a deadline, which is how one belongs in a program."""
    conn = await asyncpg.connect(database="guide")
    collected = asyncio.Queue()
    await conn.add_listener(channel, lambda con, pid, ch, payload: collected.put_nowait(payload))

    got = []
    try:
        async with asyncio.timeout(seconds):
            while len(got) < count:
                got.append(await collected.get())                   # the loop is free while we wait
    except TimeoutError:
        pass
    finally:
        await conn.close()
    return got


send("queue", "one", after=0.2)
send("queue", "two", after=0.4)
print("collected:", await listen_until("queue", count=2, seconds=5))
print("and with nothing sent:", await listen_until("queue", count=1, seconds=1))


collected: ['one', 'two']
and with nothing sent: []


`asyncio.Queue` between the callback and the waiting code is the shape worth remembering: the
callback must not block and must not be async, so it hands off and returns immediately.

### The two limits

A channel name is an identifier, which means it folds to lower case unless you quote it:


In [9]:
cased = psycopg.connect("dbname=guide", autocommit=True)
cased.execute("LISTEN MyChannel")                                   # written out, not subscribe()

print("what you typed: MyChannel")
print("what it is:    ", listening(cased))

send("MyChannel", "with the capitals")
print("notified as MyChannel:", heard(cased, seconds=1))

send("mychannel", "folded")
print("notified as mychannel:", heard(cased, seconds=2))
cased.close()


what you typed: MyChannel
what it is:     ['mychannel']
notified as MyChannel: []
notified as mychannel: [('mychannel', 'folded')]


`pg_notify` takes the channel as a string, and a string is not folded. So `LISTEN MyChannel`
subscribes to `mychannel` and `pg_notify('MyChannel', ...)` sends to `MyChannel`, and they never
meet. The rule that avoids the whole problem is to use lower case names with underscores, always.

The payload has a limit, and it is smaller than people expect:


In [10]:
sized = subscribe("sized")
for size in (7999, 8000):
    try:
        send("sized", "x" * size)
        print(f"  {size} characters: accepted")
    except errors.InvalidParameterValue as error:
        print(f"  {size} characters:", error)

print("the one that got through was", len(heard(sized, seconds=1)[0][1]), "characters long")
print("so a payload is a key, not a document")
sized.close()


  7999 characters: accepted
  8000 characters: payload string too long
the one that got through was 7999 characters long
so a payload is a key, not a document


### When to reach for which

| What you want | How to write it |
|---|---|
| to subscribe | `conn.execute("LISTEN name")`, with `autocommit=True` |
| to send, with a fixed name | `NOTIFY name, 'payload'` |
| to send, with a name in a variable | `SELECT pg_notify(%s, %s)` |
| to wait, in psycopg | `conn.notifies(timeout=..., stop_after=...)` |
| to react, in psycopg | `conn.add_notify_handler(fn)`, and not both |
| to subscribe, in asyncpg | `await conn.add_listener(name, fn)` |
| a row to be the message | an `AFTER INSERT` trigger calling `pg_notify` |
| to stop | `UNLISTEN name`, or `UNLISTEN *` |

The default is a trigger and a listener with a deadline. Reach for `notifies` when the program is
synchronous and `add_listener` when it is not, and give the listening connection nothing else to do:
it is not in the pool, it holds itself, and **Connection Pools** said so for this reason.

### A worker that does not trust the notification, finished

The shape that survives being disconnected, which is the shape to write. The notification is a
nudge; the table is the truth.


In [11]:
def drain(conn):
    """Take whatever is in the queue now, however we came to be looking."""
    with conn.transaction():
        rows = conn.execute("DELETE FROM queue RETURNING id, task").fetchall()
    return rows


def work(seconds=3):
    """Catch up first, then wait to be told, and catch up again whenever anything arrives."""
    worker = psycopg.connect("dbname=guide", autocommit=True)
    worker.execute("LISTEN queue")

    done = drain(worker)                                            # anything missed while away
    print("  caught up first:", done)

    while True:
        nudge = list(worker.notifies(timeout=seconds, stop_after=1))   # let the wait finish first
        if not nudge:
            break
        print("  nudged by:", nudge[0].payload, "-> did", drain(worker))

    worker.close()
    return done


def add_task(task, after):
    """Write a row from another connection a moment from now, and let the trigger do the rest."""
    def once():
        with psycopg.connect("dbname=guide", autocommit=True) as writer:
            writer.execute("INSERT INTO queue (task) VALUES (%s)", (task,))

    threading.Timer(after, once).start()


with psycopg.connect("dbname=guide", autocommit=True) as ahead:
    ahead.execute("DELETE FROM queue")
    ahead.execute("INSERT INTO queue (task) VALUES ('written while the worker was down')")

add_task("resize", after=0.4)                                       # while the worker is listening
add_task("thumbnail", after=0.8)
work()


  caught up first: [(2, 'written while the worker was down')]
  nudged by: 3:resize -> did [(3, 'resize')]
  nudged by: 4:thumbnail -> did [(4, 'thumbnail')]


[(2, 'written while the worker was down')]

The `list` around `notifies` is not decoration. While that generator is running it owns the
connection, so querying the same connection from inside the loop body waits for a generator that is
waiting for you. Finish the wait, then use the connection.

The first `drain` is the whole point. A worker that only ever acts on notifications loses every row
written while it was restarting, and a worker that drains on startup and drains again on every nudge
cannot, however many notifications it misses or receives twice.

**An Event Store** builds this out at the end of the guide, where a thousand rows arriving at once
produce a single notification, and this shape is the only reason that is survivable.

### Where each part came from

| In the worker | What it relies on | The section that showed it |
|---|---|---|
| `autocommit=True` | a listener in a transaction hears nothing | A first look |
| `LISTEN queue` | one name, subscribed to | A listener, and something to hear |
| the first `drain` | a notification sent while away is gone | Several channels, and which one spoke |
| `notifies(timeout=...)` | a wait that ends | A listener, and something to hear |
| `with conn.transaction()` | the delete and the work are one unit | **Transactions and Errors** |
| the trigger sending it | the writer not knowing a worker exists | A trigger, so the row is the message |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/15-listen-and-notify-solutions.ipynb).

**1.** Listen on a channel, send something to it, and print what arrived.


In [12]:
# your code here


**2.** Listen on two channels and show which one each message came from.


In [13]:
# your code here


**3.** Send a notification from inside a transaction and show it arrives only on commit.


In [14]:
# your code here


**4.** Do the same with asyncpg and `add_listener`.


In [15]:
# your code here


**5.** Show that `LISTEN Upper` does not hear `pg_notify('Upper', ...)`.


In [16]:
# your code here


**6.** Find the largest payload the server will accept.


In [17]:
# your code here


## Common errors

### psycopg.errors.InvalidParameterValue: payload string too long


In [18]:
with psycopg.connect("dbname=guide", autocommit=True) as sender:
    sender.execute("SELECT pg_notify('jobs', %s)", ("x" * 8000,))


InvalidParameterValue: payload string too long

Eight thousand bytes, of which the last is the terminator, so seven thousand nine hundred and
ninety-nine characters is the most that fits. It is a hard limit in the server and there is no
setting for it.

The fix is not a bigger payload. It is to send an identifier and let the listener read the row,
which is what a notification is for:


In [19]:
watching = subscribe("queue")
with psycopg.connect("dbname=guide", autocommit=True) as sender:
    sender.execute("INSERT INTO queue (task) VALUES ('a task with a lot to say')")

identifier = heard(watching, seconds=3)[0][1].split(":")[0]
print("the whole payload was an id:", identifier)
print("and the listener goes and reads the row:",
      watching.execute("SELECT task FROM queue WHERE id = %s", (identifier,)).fetchone())


the whole payload was an id: 5
and the listener goes and reads the row: ('a task with a lot to say',)


### RuntimeWarning: using 'notifies()' together with notifies handlers


In [20]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    listener.add_notify_handler(lambda note: print("  the handler saw:", note.payload))
    send("jobs", "which one gets this")
    heard(listener, seconds=1)

    for warning in caught:
        print(f"{warning.category.__name__}: {warning.message}")


  the handler saw: written by hand
  the handler saw: and this one goes nowhere
  the handler saw: one
  the handler saw: two
  the handler saw: 2:written while the worker was down
  the handler saw: 3:resize
  the handler saw: 4:thumbnail
  the handler saw: 5:a task with a lot to say
  the handler saw: which one gets this


Two ways of receiving the same notifications, on one connection, and psycopg says plainly that using
both is unreliable rather than silently picking one. Choose the handler for a program that is doing
other things, and `notifies` for a loop whose whole job is waiting.


In [21]:
listener.remove_notify_handler(listener._notify_handlers[0])
print("handlers now:", len(listener._notify_handlers))


handlers now: 0


### No error: the listener that was not there

A notification is not stored. Nobody listening means nobody will ever know:


In [22]:
send("nobody_home", "sent into the void")
time.sleep(0.3)

late = psycopg.connect("dbname=guide", autocommit=True)
late.execute("LISTEN nobody_home")
print("listening now, and asking for what was sent a moment ago:",
      heard(late, seconds=1))
late.close()


listening now, and asking for what was sent a moment ago: []


Nothing, and nothing will bring it back. This is not a queue and it makes no attempt to be one:
there is no acknowledgement, no redelivery and no history.

Every reliable design built on this puts the truth in a table and treats the notification as a hint
that the table changed, which is exactly what the finished worker above does.

### No error: the callback that raised


In [23]:
def explodes(conn, pid, channel, payload):
    raise RuntimeError("this callback has a bug in it")


reported = []
asyncio.get_running_loop().set_exception_handler(
    lambda loop, context: reported.append(f"{type(context['exception']).__name__}: "
                                          f"{context['exception']}"))

watcher = await asyncpg.connect(database="guide")
await watcher.add_listener("explosions", explodes)

send("explosions", "go")
await asyncio.sleep(0.5)

print("the await above raised nothing at all")
print("the event loop reported:", reported)

asyncio.get_running_loop().set_exception_handler(None)
await watcher.close()


the await above raised nothing at all
the event loop reported: ['RuntimeError: this callback has a bug in it']


asyncpg dispatches a listener callback through the event loop rather than at any line you wrote, so
an exception inside one never reaches your `await`. Without the handler installed above it prints
`Exception in callback` to the console and the program continues, missing that notification and
every side effect the callback was supposed to have.

Catch inside the callback. It is the only place the exception is yours to handle:


In [24]:
def careful(conn, pid, channel, payload):
    try:
        raise RuntimeError("the same bug")
    except RuntimeError as error:
        print("  handled where it happened:", error)


watcher = await asyncpg.connect(database="guide")
await watcher.add_listener("explosions", careful)
send("explosions", "go")
await asyncio.sleep(0.5)
await watcher.close()


  handled where it happened: the same bug


In [25]:
for closing in (listener, watching):
    closing.execute("UNLISTEN *")
    closing.close()
await conn.close()
print("listeners closed")


listeners closed


## Recap

- `LISTEN name` subscribes a session, `NOTIFY name, 'payload'` and `pg_notify(name, payload)` send.
  A notification is delivered on commit and discarded if nobody is listening.
- A psycopg listener needs `autocommit=True`. Without it the connection sits in a transaction and
  hears nothing, with no error anywhere.
- Wait with `conn.notifies(timeout=..., stop_after=...)`, or react with `add_notify_handler`, and
  never both on one connection.
- asyncpg uses `await conn.add_listener(channel, callback)`. The callback is called by the event
  loop, must not block, and its exceptions never reach your `await`.
- A channel written in a `LISTEN` statement folds to lower case; the same name passed to `pg_notify`
  does not. Use lower case names.
- A payload is at most 7999 characters. Send an identifier and let the listener read the row.
- The reliable shape is drain, then listen, then drain again on every nudge. The table is the truth
  and the notification is only a hint that it changed.


## What is next

**Which Driver** is the comparison the guide has been deferring: psycopg and asyncpg on one
workload, measured three ways, and the missing `await` that makes an asynchronous benchmark look
thousands of times faster than it is.


---

&#8592; **Previous:** [Connecting to a Hosted Server](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/14-connecting-to-a-hosted-server.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
